In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [11]:
from pyspark.sql.functions import *

# Revenue Gap Analysis

**Difficulty:** Hard
**Type:** Pandas / PySpark
**Tags:** `sql` `pandas` `pyspark` `date-functions` `gaps-and-islands` `complex-logic`

---

## Problem Statement

You are given a DataFrame:

### `daily_revenue`

| Column | Type | Description |
|---|---|---|
| `store_id` | int | Store identifier |
| `revenue_date` | str | Date of revenue (`YYYY-MM-DD`) |
| `revenue` | float | Revenue amount for that day |

### Task

Find periods where a store had **zero revenue for 3 or more consecutive days**. A day with zero revenue includes:

- Days explicitly present in the data with `revenue = 0`, **and**
- Days **missing** from the data entirely (implicit zeros)

For each qualifying gap, report:

| Column | Description |
|---|---|
| `store_id` | The store |
| `gap_start` | First date of the zero-revenue period |
| `gap_end` | Last date of the zero-revenue period |
| `gap_days` | Number of consecutive zero-revenue days |
| `revenue_before_gap` | Total revenue in the 7 days immediately before `gap_start` |
| `revenue_after_gap` | Total revenue in the 7 days immediately after `gap_end` |

Return results sorted by:
1. `gap_days` **DESC**
2. `store_id` **ASC**
3. `gap_start` **ASC**

---

## Example

Store 1 has revenue on Jan 1–3, no revenue Jan 4–7, revenue again Jan 8+.

**Result:** `store_id=1, gap_start="2023-01-04", gap_end="2023-01-07", gap_days=4`

---

## Hints

1. Generate the full date range for each store (from min to max date), then left join with actual revenue data. Missing dates get `revenue = 0`.
2. Use a **gaps-and-islands** technique: mark zero-revenue days, create groups of consecutive zeros, then filter groups with 3+ days.
3. For `revenue_before_gap`, sum revenue in the 7 days strictly *before* `gap_start`. For `revenue_after_gap`, sum revenue in the 7 days strictly *after* `gap_end`.

---

## Sample Test Case

### Input — `daily_revenue`

| store_id | revenue_date | revenue |
|---|---|---|
| 1 | 2023-01-01 | 500.0 |
| 1 | 2023-01-02 | 600.0 |
| 1 | 2023-01-03 | 450.0 |
| 1 | 2023-01-07 | 700.0 |
| 1 | 2023-01-08 | 550.0 |
| 1 | 2023-01-09 | 800.0 |
| 2 | 2023-01-01 | 300.0 |
| 2 | 2023-01-02 | 350.0 |
| 2 | 2023-01-06 | 400.0 |
| 2 | 2023-01-07 | 420.0 |

### Expected Output

| store_id | gap_start | gap_end | gap_days | revenue_before_gap | revenue_after_gap |
|---|---|---|---|---|---|
| 1 | 2023-01-04 | 2023-01-06 | 3 | 1550.0 | 2050.0 |
| 2 | 2023-01-03 | 2023-01-05 | 3 | 650.0 | 820.0 |

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("store_id", IntegerType(), False),
    StructField("revenue_date", StringType(), False),
    StructField("revenue", DoubleType(), False),
])

data = [
    (1, "2023-01-01", 500.0),
    (1, "2023-01-02", 600.0),
    (1, "2023-01-03", 450.0),
    (1, "2023-01-07", 700.0),
    (1, "2023-01-08", 550.0),
    (1, "2023-01-09", 800.0),
    (2, "2023-01-01", 300.0),
    (2, "2023-01-02", 350.0),
    (2, "2023-01-06", 400.0),
    (2, "2023-01-07", 420.0),
]

daily_revenue = spark.createDataFrame(data, schema=schema)
daily_revenue.show()

+--------+------------+-------+
|store_id|revenue_date|revenue|
+--------+------------+-------+
|       1|  2023-01-01|  500.0|
|       1|  2023-01-02|  600.0|
|       1|  2023-01-03|  450.0|
|       1|  2023-01-07|  700.0|
|       1|  2023-01-08|  550.0|
|       1|  2023-01-09|  800.0|
|       2|  2023-01-01|  300.0|
|       2|  2023-01-02|  350.0|
|       2|  2023-01-06|  400.0|
|       2|  2023-01-07|  420.0|
+--------+------------+-------+



In [5]:
daily_revenue.createOrReplaceTempView("revenue")

# Using SQL

In [7]:
spark.sql("SELECT * from revenue").show()

+--------+------------+-------+
|store_id|revenue_date|revenue|
+--------+------------+-------+
|       1|  2023-01-01|  500.0|
|       1|  2023-01-02|  600.0|
|       1|  2023-01-03|  450.0|
|       1|  2023-01-07|  700.0|
|       1|  2023-01-08|  550.0|
|       1|  2023-01-09|  800.0|
|       2|  2023-01-01|  300.0|
|       2|  2023-01-02|  350.0|
|       2|  2023-01-06|  400.0|
|       2|  2023-01-07|  420.0|
+--------+------------+-------+



# Using Pyspark

In [34]:
date_range_df = daily_revenue.\
                    groupBy(col("store_id"))\
                    .agg(
                        min(to_date(col("revenue_date"))).alias("min_date"),
                        max(to_date(col("revenue_date"))).alias("max_date")
                    )

In [35]:
date_df = date_range_df\
          .withColumn("dates",
                        explode(sequence(col("min_date"),col("max_date"))
                               )
                       )\
            .select(col("store_id"),
                    col("dates").cast("string").alias("revenue_date")
                   )

In [40]:
full_revenue_df = date_df\
    .join(
        daily_revenue,on = ["store_id","revenue_date"],how = "left"
        )\
    .withColumn(
        "revenue",coalesce(
                            col("revenue"),lit("0.0")
                          )
    ).orderBy("store_id","revenue_date")

In [51]:
from pyspark.sql.window import Window

window_specs = Window.partitionBy(col("store_id")).orderBy(col("revenue_date"))

full_revenue_df.withColumn("prev_date",lag(col("revenue_date").over(window_specs))).show()

AnalysisException: [UNSUPPORTED_EXPR_FOR_WINDOW] Expression "revenue_date" not supported within a window function.;
Project [store_id#19, revenue_date#244, revenue#348, prev_date#433]
+- Project [store_id#19, revenue_date#244, revenue#348, _w0#434, lag(_w0#434, -1, null) AS prev_date#433]
   +- Project [store_id#19, revenue_date#244, revenue#348, _w0#434]
      +- Project [store_id#19, revenue_date#244, revenue#348, _w0#434, _w0#434]
         +- Window [revenue_date#244 windowspecdefinition(store_id#19, revenue_date#244 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())) AS _w0#434], [store_id#19], [revenue_date#244 ASC NULLS FIRST]
            +- Project [store_id#19, revenue_date#244, revenue#348]
               +- Sort [store_id#19 ASC NULLS FIRST, revenue_date#244 ASC NULLS FIRST], true
                  +- Project [store_id#19, revenue_date#244, coalesce(cast(revenue#344 as string), 0.0) AS revenue#348]
                     +- Project [store_id

In [49]:
full_revenue_df.show()

+--------+------------+-------+
|store_id|revenue_date|revenue|
+--------+------------+-------+
|       1|  2023-01-01|  500.0|
|       1|  2023-01-02|  600.0|
|       1|  2023-01-03|  450.0|
|       1|  2023-01-04|    0.0|
|       1|  2023-01-05|    0.0|
|       1|  2023-01-06|    0.0|
|       1|  2023-01-07|  700.0|
|       1|  2023-01-08|  550.0|
|       1|  2023-01-09|  800.0|
|       2|  2023-01-01|  300.0|
|       2|  2023-01-02|  350.0|
|       2|  2023-01-03|    0.0|
|       2|  2023-01-04|    0.0|
|       2|  2023-01-05|    0.0|
|       2|  2023-01-06|  400.0|
|       2|  2023-01-07|  420.0|
+--------+------------+-------+

